# Random Forest



## Fetch Data

In [40]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("camnugent/california-housing-prices")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'california-housing-prices' dataset.
Path to dataset files: /kaggle/input/california-housing-prices


In [41]:
!ls $path

housing.csv


In [42]:
import pandas as pd
import numpy as np

df = pd.read_csv(path+"/housing.csv")

In [43]:
df.head(3)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY


In [44]:
df.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value', 'ocean_proximity'],
      dtype='object')

In [45]:
rooms_per_household = df['total_rooms']/df['households']
df['rooms_per_household'] = rooms_per_household

In [46]:
df = df.drop("households", axis='columns')

In [47]:
df = df.drop("total_rooms", axis='columns')

In [48]:
df.head(3)

,longitude,latitude,housing_median_age,total_bedrooms,population,median_income,median_house_value,ocean_proximity,rooms_per_household
0,-122.23,37.88,41.0,129.0,322.0,8.3252,452600.0,NEAR BAY,6.984127
1,-122.22,37.86,21.0,1106.0,2401.0,8.3014,358500.0,NEAR BAY,6.238137
2,-122.24,37.85,52.0,190.0,496.0,7.2574,352100.0,NEAR BAY,8.288136


In [49]:
df = pd.get_dummies(df, columns=['ocean_proximity'])

In [50]:
df.head()

,longitude,latitude,housing_median_age,total_bedrooms,population,median_income,median_house_value,rooms_per_household,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.23,37.88,41.0,129.0,322.0,8.3252,452600.0,6.984127,False,False,False,True,False
1,-122.22,37.86,21.0,1106.0,2401.0,8.3014,358500.0,6.238137,False,False,False,True,False
2,-122.24,37.85,52.0,190.0,496.0,7.2574,352100.0,8.288136,False,False,False,True,False
3,-122.25,37.85,52.0,235.0,558.0,5.6431,341300.0,5.817352,False,False,False,True,False
4,-122.25,37.85,52.0,280.0,565.0,3.8462,342200.0,6.281853,False,False,False,True,False


In [51]:
df.columns = df.columns.str.replace(" ", "_")

In [52]:
df.head(3)

,longitude,latitude,housing_median_age,total_bedrooms,population,median_income,median_house_value,rooms_per_household,ocean_proximity_<1H_OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR_BAY,ocean_proximity_NEAR_OCEAN
0,-122.23,37.88,41.0,129.0,322.0,8.3252,452600.0,6.984127,False,False,False,True,False
1,-122.22,37.86,21.0,1106.0,2401.0,8.3014,358500.0,6.238137,False,False,False,True,False
2,-122.24,37.85,52.0,190.0,496.0,7.2574,352100.0,8.288136,False,False,False,True,False


In [53]:
df.isnull().sum()

,0
longitude,0
latitude,0
housing_median_age,0
total_bedrooms,207
population,0
median_income,0
median_house_value,0
rooms_per_household,0
ocean_proximity_<1H_OCEAN,0
ocean_proximity_INLAND,0


In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   longitude                   20640 non-null  float64
 1   latitude                    20640 non-null  float64
 2   housing_median_age          20640 non-null  float64
 3   total_bedrooms              20433 non-null  float64
 4   population                  20640 non-null  float64
 5   median_income               20640 non-null  float64
 6   median_house_value          20640 non-null  float64
 7   rooms_per_household         20640 non-null  float64
 8   ocean_proximity_<1H_OCEAN   20640 non-null  bool   
 9   ocean_proximity_INLAND      20640 non-null  bool   
 10  ocean_proximity_ISLAND      20640 non-null  bool   
 11  ocean_proximity_NEAR_BAY    20640 non-null  bool   
 12  ocean_proximity_NEAR_OCEAN  20640 non-null  bool   
dtypes: bool(5), float64(8)
memory u

In [61]:
## imputer
from sklearn.impute import KNNImputer



In [63]:
imputer = KNNImputer(n_neighbors=5)
numeric_cols = ['housing_median_age', 'total_bedrooms', 'population', 'median_income', 'rooms_per_household']
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

In [64]:
df.head(3)

,longitude,latitude,housing_median_age,total_bedrooms,population,median_income,median_house_value,rooms_per_household,ocean_proximity_<1H_OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR_BAY,ocean_proximity_NEAR_OCEAN
0,-122.23,37.88,41.0,129.0,322.0,8.3252,452600.0,6.984127,False,False,False,True,False
1,-122.22,37.86,21.0,1106.0,2401.0,8.3014,358500.0,6.238137,False,False,False,True,False
2,-122.24,37.85,52.0,190.0,496.0,7.2574,352100.0,8.288136,False,False,False,True,False


In [65]:
df.isnull().sum()

,0
longitude,0
latitude,0
housing_median_age,0
total_bedrooms,0
population,0
median_income,0
median_house_value,0
rooms_per_household,0
ocean_proximity_<1H_OCEAN,0
ocean_proximity_INLAND,0


In [66]:
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

In [67]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [75]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(
    n_estimators=1000,
    max_depth=None,
    random_state=42,
)

model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=1000, random_state=42)

In [76]:
y_pred = model.predict(X_test)

In [77]:
y_pred[:6]

array([ 57657.1  ,  61223.8  , 458693.866, 257842.103, 261216.9  ,
       158114.4  ])

In [78]:
y_test[:6]

,median_house_value
20046,47700.0
3024,45800.0
15663,500001.0
20484,218600.0
9814,278000.0
13311,158700.0
